# 🧍 Bad Posture Detection — YOLOv8 Classification

**Dataset**: Roboflow `posture-classification-iqdle` (2 kelas: `bad`, `good`)  
**Model**: YOLOv8n-cls & YOLOv8s-cls (Ultralytics)  
**GPU**: Kaggle P100  

### Pipeline
```
Roboflow ZIP  →  Dataset Check  →  EDA  →  Train YOLOv8-cls
→  Evaluasi (CM, PR, F1)  →  Inspeksi Prediksi Salah
→  Export ONNX  →  Simulasi Inference Real-time
```

### Cara upload dataset ke Kaggle
1. Download ZIP dari Roboflow (format: **YOLOv8** atau **Folder Structure**)
2. Kaggle → Datasets → New Dataset → Upload ZIP
3. Tambahkan dataset ke notebook ini via **+ Add Data**
4. Ubah `DATASET_ROOT` di Sel 2 sesuai path yang muncul

## Sel 1 — Install & Import

In [ ]:
# YOLOv8 via ultralytics (sudah include di Kaggle, tapi pastikan versi terbaru)
!pip install ultralytics --upgrade --quiet

import os, shutil, random, glob, json, warnings
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from PIL import Image
import torch

from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_curve
)

warnings.filterwarnings('ignore')

# ── GPU check ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Ultralytics siap')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {device}', end='')
if device == 'cuda':
    print(f' ({torch.cuda.get_device_name(0)})')
else:
    print(' ⚠️  GPU tidak ditemukan — training akan lambat')

## Sel 2 — Konfigurasi

In [ ]:
# ─── SESUAIKAN PATH INI ────────────────────────────────────────────────────
# Cek path dataset kamu dengan: !ls /kaggle/input/
DATASET_ROOT = '/kaggle/input/posture-classification-iqdle'  # <-- ubah jika perlu
# ───────────────────────────────────────────────────────────────────────────

WORK_DIR     = Path('/kaggle/working')
RUN_DIR      = WORK_DIR / 'runs'
EXPORT_DIR   = WORK_DIR / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES  = ['bad', 'good']          # urutan abjad = default YOLOv8
CLASS_COLORS = {'bad': '#D85A30', 'good': '#1D9E75'}
IMG_SIZE     = 224                      # resolusi training
BATCH_SIZE   = 32
EPOCHS_NANO  = 50                       # YOLOv8n-cls (cepat, baseline)
EPOCHS_SMALL = 60                       # YOLOv8s-cls (lebih akurat)
PATIENCE     = 15                       # early stopping
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print('✅ Konfigurasi selesai')
print(f'   Dataset root : {DATASET_ROOT}')
print(f'   Classes      : {CLASS_NAMES}')
print(f'   Image size   : {IMG_SIZE}px')
print(f'   Batch size   : {BATCH_SIZE}')

# Cek apakah dataset tersedia
if not os.path.exists(DATASET_ROOT):
    print(f'\n⚠️  Path tidak ditemukan: {DATASET_ROOT}')
    print('   Jalankan: !ls /kaggle/input/ untuk melihat nama dataset')
else:
    print(f'\n   Isi dataset root:')
    for item in sorted(os.listdir(DATASET_ROOT)):
        print(f'   ├── {item}')

## Sel 3 — Inspeksi Struktur Dataset

Roboflow ZIP biasanya menghasilkan struktur:
```
dataset-root/
  train/ ── bad/ , good/
  valid/ ── bad/ , good/
  test/  ── bad/ , good/
```
Sel ini akan mendeteksi struktur tersebut secara otomatis.

In [ ]:
def count_images(folder: str) -> dict:
    """Hitung gambar per kelas dalam satu split folder."""
    counts = {}
    exts   = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    if not os.path.isdir(folder):
        return counts
    for cls in sorted(os.listdir(folder)):
        cls_path = os.path.join(folder, cls)
        if os.path.isdir(cls_path):
            n = sum(
                1 for f in os.listdir(cls_path)
                if Path(f).suffix.lower() in exts
            )
            counts[cls] = n
    return counts


splits = ['train', 'valid', 'test']
stats  = {}
print('=== Distribusi Dataset ===')
print(f'{"Split":<10} {"bad":>8} {"good":>8} {"total":>8}')
print('-' * 38)

for split in splits:
    path   = os.path.join(DATASET_ROOT, split)
    counts = count_images(path)
    stats[split] = counts
    total  = sum(counts.values())
    bad_n  = counts.get('bad',  0)
    good_n = counts.get('good', 0)
    print(f'{split:<10} {bad_n:>8} {good_n:>8} {total:>8}')

print('-' * 38)
total_all = sum(sum(v.values()) for v in stats.values())
print(f'{"TOTAL":<10} '
      f'{sum(s.get("bad",0) for s in stats.values()):>8} '
      f'{sum(s.get("good",0) for s in stats.values()):>8} '
      f'{total_all:>8}')

# Cek class imbalance
train_counts = stats.get('train', {})
if len(train_counts) == 2:
    ratio = max(train_counts.values()) / (min(train_counts.values()) + 1e-6)
    print(f'\nClass imbalance ratio (train): {ratio:.2f}x')
    if ratio > 2:
        print('⚠️  Imbalance cukup besar — pertimbangkan augmentasi lebih kuat pada kelas minoritas')
    else:
        print('✅ Dataset relatif seimbang')

## Sel 4 — EDA: Visualisasi Sampel Gambar

In [ ]:
def load_sample_images(split: str, cls: str, n: int = 6) -> list:
    """Ambil n gambar acak dari kelas tertentu."""
    folder = os.path.join(DATASET_ROOT, split, cls)
    if not os.path.isdir(folder):
        return []
    exts  = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    files = [f for f in os.listdir(folder) if Path(f).suffix.lower() in exts]
    chosen = random.sample(files, min(n, len(files)))
    imgs = []
    for f in chosen:
        img = cv2.imread(os.path.join(folder, f))
        if img is not None:
            imgs.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    return imgs


n_cols    = 6
bad_imgs  = load_sample_images('train', 'bad',  n_cols)
good_imgs = load_sample_images('train', 'good', n_cols)
n_rows    = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5.5))
fig.suptitle('Sampel Dataset Training — bad (atas) vs good (bawah)',
             fontsize=14, fontweight='bold', y=1.01)
fig.patch.set_facecolor('#0d0f14')

for row, (imgs, label) in enumerate([(bad_imgs, 'bad'), (good_imgs, 'good')]):
    color  = CLASS_COLORS[label]
    for col in range(n_cols):
        ax = axes[row][col]
        ax.set_facecolor('#0d0f14')
        if col < len(imgs):
            ax.imshow(imgs[col])
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
        else:
            ax.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(label.upper(), color=color, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(WORK_DIR / 'eda_samples.png', dpi=120,
            bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('Gambar tersimpan: eda_samples.png')

In [ ]:
# Distribusi ukuran gambar dan resolusi
def sample_image_stats(split: str, cls: str, n: int = 50) -> list:
    folder = os.path.join(DATASET_ROOT, split, cls)
    if not os.path.isdir(folder):
        return []
    exts  = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    files = [f for f in os.listdir(folder) if Path(f).suffix.lower() in exts]
    chosen = random.sample(files, min(n, len(files)))
    stats = []
    for f in chosen:
        try:
            img = Image.open(os.path.join(folder, f))
            w, h = img.size
            stats.append({'cls': cls, 'width': w, 'height': h,
                          'aspect': w/h, 'size_kb': os.path.getsize(
                              os.path.join(folder, f)) / 1024})
        except Exception:
            pass
    return stats


all_stats = []
for cls in CLASS_NAMES:
    all_stats.extend(sample_image_stats('train', cls, 80))
df_stats = pd.DataFrame(all_stats)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Statistik Ukuran Gambar (sample train)', fontsize=13, fontweight='bold')

for cls, grp in df_stats.groupby('cls'):
    col = CLASS_COLORS.get(cls, '#888')
    axes[0].hist(grp['width'],  bins=20, alpha=0.7, label=cls, color=col)
    axes[1].hist(grp['height'], bins=20, alpha=0.7, label=cls, color=col)
    axes[2].hist(grp['aspect'], bins=20, alpha=0.7, label=cls, color=col)

axes[0].set(title='Width (px)',  xlabel='px', ylabel='Count')
axes[1].set(title='Height (px)', xlabel='px')
axes[2].set(title='Aspect Ratio (W/H)', xlabel='ratio')
for ax in axes:
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'eda_image_stats.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nStatistik ringkas:')
display(df_stats.groupby('cls')[['width','height','aspect','size_kb']].describe().round(1))

## Sel 5 — Training: YOLOv8n-cls (Baseline)

YOLOv8n-cls (nano) dipilih sebagai baseline karena:
- Hanya 2.7M parameter → training selesai ~5–10 menit di P100
- Pretrained ImageNet → cocok untuk fine-tuning
- Hasil biasanya sudah >90% akurasi untuk 2-kelas sederhana

In [ ]:
print('=' * 55)
print('  Training YOLOv8n-cls (Nano — Baseline)')
print('=' * 55)

model_nano = YOLO('yolov8n-cls.pt')   # download pretrained otomatis

results_nano = model_nano.train(
    data    = DATASET_ROOT,           # folder root dengan train/valid/test
    epochs  = EPOCHS_NANO,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = device,
    patience= PATIENCE,               # early stopping
    seed    = RANDOM_SEED,
    project = str(RUN_DIR),
    name    = 'nano',
    workers = 2,
    # ── Augmentasi ──────────────────────────────────────────
    hsv_h   = 0.015,   # hue jitter
    hsv_s   = 0.5,     # saturation jitter
    hsv_v   = 0.4,     # brightness jitter
    fliplr  = 0.5,     # flip horizontal 50%
    degrees = 10,      # rotasi ±10°
    scale   = 0.3,     # zoom ±30%
    translate= 0.1,    # translasi ±10%
    erasing = 0.2,     # random erasing
    # ── Optimizer ───────────────────────────────────────────
    optimizer='AdamW',
    lr0     = 1e-3,
    lrf     = 0.01,    # learning rate final = lr0 * lrf
    warmup_epochs = 3,
    cos_lr  = True,    # cosine LR scheduler
    # ── Logging ─────────────────────────────────────────────
    plots   = True,    # simpan grafik training otomatis
    save    = True,
    verbose = True,
)

NANO_PATH = str(RUN_DIR / 'nano' / 'weights' / 'best.pt')
print(f'\n✅ Model nano tersimpan: {NANO_PATH}')

## Sel 6 — Training: YOLOv8s-cls (Small — Better Accuracy)

YOLOv8s-cls punya kapasitas 2.4× lebih besar dari nano — biasanya +2–5% akurasi
dengan waktu training ~2× lebih lama (~10–18 menit di P100).

In [ ]:
print('=' * 55)
print('  Training YOLOv8s-cls (Small — Higher Accuracy)')
print('=' * 55)

model_small = YOLO('yolov8s-cls.pt')

results_small = model_small.train(
    data    = DATASET_ROOT,
    epochs  = EPOCHS_SMALL,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = device,
    patience= PATIENCE,
    seed    = RANDOM_SEED,
    project = str(RUN_DIR),
    name    = 'small',
    workers = 2,
    # Augmentasi (sama seperti nano)
    hsv_h   = 0.015, hsv_s = 0.5, hsv_v = 0.4,
    fliplr  = 0.5,   degrees = 10, scale = 0.3,
    translate= 0.1,  erasing = 0.2,
    # Optimizer (sedikit lebih konservatif untuk small)
    optimizer='AdamW',
    lr0     = 5e-4,
    lrf     = 0.01,
    warmup_epochs = 3,
    cos_lr  = True,
    plots   = True,
    save    = True,
    verbose = True,
)

SMALL_PATH = str(RUN_DIR / 'small' / 'weights' / 'best.pt')
print(f'\n✅ Model small tersimpan: {SMALL_PATH}')

## Sel 7 — Visualisasi Training Curves

In [ ]:
def load_results_csv(run_name: str) -> pd.DataFrame:
    """Load results.csv yang di-generate Ultralytics selama training."""
    csv_path = RUN_DIR / run_name / 'results.csv'
    if not csv_path.exists():
        print(f'  results.csv tidak ditemukan: {csv_path}')
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # hapus spasi di nama kolom
    return df


df_nano  = load_results_csv('nano')
df_small = load_results_csv('small')

def plot_training_curves(df: pd.DataFrame, run_name: str, color: str):
    """Plot loss dan accuracy curves dalam satu figure."""
    if df.empty:
        print(f'  Tidak ada data untuk {run_name}')
        return

    # Deteksi nama kolom dinamis (berbeda versi Ultralytics)
    cols = df.columns.tolist()
    train_loss_col = next((c for c in cols if 'train' in c.lower() and 'loss' in c.lower()), None)
    val_loss_col   = next((c for c in cols if 'val'   in c.lower() and 'loss' in c.lower()), None)
    top1_col       = next((c for c in cols if 'top1'  in c.lower() or 'acc'   in c.lower()), None)
    top5_col       = next((c for c in cols if 'top5'  in c.lower()), None)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(f'Training Curves — {run_name}',
                 fontsize=13, fontweight='bold')

    epochs = df.get('epoch', range(len(df)))

    # Loss plot
    ax = axes[0]
    if train_loss_col:
        ax.plot(epochs, df[train_loss_col], label='Train Loss',
                color=color, linewidth=2)
    if val_loss_col:
        ax.plot(epochs, df[val_loss_col], label='Val Loss',
                color=color, linewidth=2, linestyle='--', alpha=0.7)
    ax.set(title='Loss', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # Accuracy plot
    ax = axes[1]
    if top1_col:
        ax.plot(epochs, df[top1_col], label='Top-1 Acc',
                color=color, linewidth=2)
        best_epoch = df[top1_col].idxmax()
        best_val   = df[top1_col].max()
        ax.axvline(df.iloc[best_epoch]['epoch'] if 'epoch' in df.columns else best_epoch,
                   color='gold', linestyle=':', linewidth=1.5, label=f'Best {best_val:.4f}')
    if top5_col:
        ax.plot(epochs, df[top5_col], label='Top-5 Acc',
                color=color, linewidth=1.5, linestyle='--', alpha=0.7)
    ax.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    save_path = WORK_DIR / f'curves_{run_name}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  Tersimpan: {save_path}')


plot_training_curves(df_nano,  'nano',  '#BA7517')
plot_training_curves(df_small, 'small', '#1D9E75')

## Sel 8 — Pilih Model Terbaik & Evaluasi pada Test Set

In [ ]:
def get_best_top1(run_name: str) -> float:
    df = load_results_csv(run_name)
    if df.empty:
        return 0.0
    top1_col = next((c for c in df.columns if 'top1' in c.lower() or
                     ('acc' in c.lower() and 'val' in c.lower())), None)
    return float(df[top1_col].max()) if top1_col else 0.0


nano_acc  = get_best_top1('nano')
small_acc = get_best_top1('small')

print(f'YOLOv8n-cls best val Top-1: {nano_acc:.4f}')
print(f'YOLOv8s-cls best val Top-1: {small_acc:.4f}')

if small_acc >= nano_acc:
    BEST_MODEL_PATH = SMALL_PATH
    BEST_MODEL_NAME = 'YOLOv8s-cls'
else:
    BEST_MODEL_PATH = NANO_PATH
    BEST_MODEL_NAME = 'YOLOv8n-cls'

print(f'\n🏆 Model terpilih : {BEST_MODEL_NAME}')
print(f'   Weight path    : {BEST_MODEL_PATH}')

# Load dan validasi pada test set
best_model = YOLO(BEST_MODEL_PATH)
print(f'\n── Evaluasi pada TEST SET ──')
val_results = best_model.val(
    data   = DATASET_ROOT,
    split  = 'test',
    imgsz  = IMG_SIZE,
    batch  = BATCH_SIZE,
    device = device,
    plots  = False,
    verbose= True,
)

## Sel 9 — Confusion Matrix & Classification Report (Detail)

In [ ]:
def run_inference_on_split(model, split: str = 'test') -> tuple:
    """
    Jalankan inference pada seluruh gambar di split tertentu.
    Return (y_true, y_pred, y_probs, image_paths)
    """
    y_true, y_pred, y_probs, paths = [], [], [], []
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_folder = os.path.join(DATASET_ROOT, split, cls_name)
        if not os.path.isdir(cls_folder):
            continue
        img_files = [
            os.path.join(cls_folder, f)
            for f in os.listdir(cls_folder)
            if Path(f).suffix.lower() in exts
        ]
        for img_path in img_files:
            res = model.predict(img_path, imgsz=IMG_SIZE, device=device,
                                verbose=False)
            if not res:
                continue
            r          = res[0]
            probs_arr  = r.probs.data.cpu().numpy()
            pred_idx   = int(probs_arr.argmax())
            y_true.append(cls_idx)
            y_pred.append(pred_idx)
            y_probs.append(probs_arr)
            paths.append(img_path)

    return (
        np.array(y_true),
        np.array(y_pred),
        np.array(y_probs),
        paths
    )


print('Menjalankan inference pada test set... (mungkin 1-3 menit)')
y_true, y_pred, y_probs, img_paths = run_inference_on_split(best_model, 'test')

# Classification report
print('\n=== Classification Report (Test Set) ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

acc = (y_true == y_pred).mean()
print(f'Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)')

In [ ]:
# Confusion matrix + ROC curve berdampingan
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Evaluasi Test Set — {BEST_MODEL_NAME}  (Acc: {acc:.4f})',
             fontsize=13, fontweight='bold')

# ── Confusion Matrix ────────────────────────────────────────────────────────
cm   = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=12)

# ── ROC Curve ──────────────────────────────────────────────────────────────
# Kelas positif = 'bad' (idx 0)
fpr, tpr, _ = roc_curve(y_true, y_probs[:, 0], pos_label=0)
roc_auc     = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#D85A30', linewidth=2.5,
             label=f'ROC AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#D85A30')
axes[1].set(title='ROC Curve (class: bad)',
            xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[1].legend(fontsize=11); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'eval_cm_roc.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall Curve
fig, ax = plt.subplots(figsize=(7, 5))
precision, recall, _ = precision_recall_curve(y_true, y_probs[:, 0], pos_label=0)
pr_auc = auc(recall, precision)
ax.plot(recall, precision, color='#1D9E75', linewidth=2.5,
        label=f'PR AUC = {pr_auc:.4f}')
ax.fill_between(recall, precision, alpha=0.12, color='#1D9E75')
baseline = (y_true == 0).mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
           alpha=0.5, label=f'Baseline ({baseline:.2f})')
ax.set(title='Precision-Recall Curve (class: bad)',
       xlabel='Recall', ylabel='Precision', xlim=[0,1], ylim=[0,1.05])
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / 'eval_pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## Sel 10 — Inspeksi Prediksi Salah (Error Analysis)

In [ ]:
# Cari indeks prediksi salah
wrong_idx = np.where(y_true != y_pred)[0]
print(f'Total prediksi salah: {len(wrong_idx)} / {len(y_true)} '
      f'({len(wrong_idx)/len(y_true)*100:.1f}%)')

# Pilah berdasarkan tipe kesalahan
false_pos = [i for i in wrong_idx if y_pred[i] == 0]  # diprediksi bad, asli good
false_neg = [i for i in wrong_idx if y_pred[i] == 1]  # diprediksi good, asli bad
print(f'  False Positive (diprediksi BAD, asli GOOD): {len(false_pos)}')
print(f'  False Negative (diprediksi GOOD, asli BAD): {len(false_neg)}')


def show_errors(indices: list, title: str, n: int = 8):
    """Tampilkan grid gambar yang salah diprediksi."""
    if not indices:
        print(f'  Tidak ada {title}')
        return
    chosen = random.sample(indices, min(n, len(indices)))
    n_show = len(chosen)
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.2, 2.8))
    if n_show == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=11, fontweight='bold', color='#D85A30')

    for ax, i in zip(axes, chosen):
        img = cv2.cvtColor(cv2.imread(img_paths[i]), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (112, 112))
        ax.imshow(img)
        conf = y_probs[i][y_pred[i]]
        true_lbl = CLASS_NAMES[y_true[i]]
        pred_lbl = CLASS_NAMES[y_pred[i]]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl}\nConf: {conf:.2f}',
                     fontsize=8, color='#D85A30')
        for spine in ax.spines.values():
            spine.set_edgecolor('#D85A30'); spine.set_linewidth(2)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    fname = title.replace(' ', '_').replace('(', '').replace(')', '') + '.png'
    plt.savefig(WORK_DIR / fname, dpi=100, bbox_inches='tight')
    plt.show()


show_errors(false_neg, 'False Negative — Diprediksi GOOD, Asli BAD (bahaya!)')
show_errors(false_pos, 'False Positive — Diprediksi BAD, Asli GOOD')

## Sel 11 — Confidence Distribution Analysis

In [ ]:
# Distribusi confidence untuk prediksi benar vs salah
correct_confs = y_probs[np.arange(len(y_pred)), y_pred][y_true == y_pred]
wrong_confs   = y_probs[np.arange(len(y_pred)), y_pred][y_true != y_pred]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Distribusi Confidence Score', fontsize=13, fontweight='bold')

axes[0].hist(correct_confs, bins=30, color='#1D9E75', alpha=0.85,
             edgecolor='white', linewidth=0.5)
axes[0].set(title=f'Prediksi BENAR (n={len(correct_confs)})',
            xlabel='Confidence', ylabel='Count')
axes[0].axvline(correct_confs.mean(), color='white', linestyle='--',
                label=f'Mean: {correct_confs.mean():.3f}')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(wrong_confs, bins=30, color='#D85A30', alpha=0.85,
             edgecolor='white', linewidth=0.5)
axes[1].set(title=f'Prediksi SALAH (n={len(wrong_confs)})',
            xlabel='Confidence')
if len(wrong_confs) > 0:
    axes[1].axvline(wrong_confs.mean(), color='white', linestyle='--',
                    label=f'Mean: {wrong_confs.mean():.3f}')
    axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Confidence benar  — mean: {correct_confs.mean():.4f},  std: {correct_confs.std():.4f}')
if len(wrong_confs) > 0:
    print(f'Confidence salah  — mean: {wrong_confs.mean():.4f},  std: {wrong_confs.std():.4f}')

## Sel 12 — Export ke ONNX (untuk Deploy di Streamlit)

In [ ]:
print('Mengekspor model ke ONNX...')

onnx_path = best_model.export(
    format  = 'onnx',
    imgsz   = IMG_SIZE,
    dynamic = True,         # support batch size dinamis
    simplify= True,         # sederhanakan graph ONNX
    opset   = 17,
)

# Salin ke output folder
import shutil
if onnx_path and os.path.exists(str(onnx_path)):
    dest = EXPORT_DIR / 'posture_best.onnx'
    shutil.copy(str(onnx_path), str(dest))
    size_mb = dest.stat().st_size / 1024 / 1024
    print(f'✅ ONNX tersimpan: {dest}  ({size_mb:.2f} MB)')

# Salin juga .pt ke output
dest_pt = EXPORT_DIR / 'posture_best.pt'
shutil.copy(BEST_MODEL_PATH, str(dest_pt))
size_pt = dest_pt.stat().st_size / 1024 / 1024
print(f'✅ PyTorch weight tersimpan: {dest_pt}  ({size_pt:.2f} MB)')

# Simpan metadata kelas
meta = {
    'model_name' : BEST_MODEL_NAME,
    'class_names': CLASS_NAMES,
    'img_size'   : IMG_SIZE,
    'accuracy'   : round(float(acc), 6),
    'roc_auc'    : round(float(roc_auc), 6),
}
(EXPORT_DIR / 'metadata.json').write_text(json.dumps(meta, indent=2))
print(f'✅ Metadata tersimpan: {EXPORT_DIR / "metadata.json"}')
print(json.dumps(meta, indent=2))

## Sel 13 — Simulasi Inference Real-time pada Test Images

Mensimulasikan bagaimana model akan dipakai di Streamlit:
setiap gambar diproses satu per satu dan divisualisasikan beserta label + confidence.

In [ ]:
def draw_posture_overlay(img_bgr: np.ndarray, label: str,
                         confidence: float, probs: dict) -> np.ndarray:
    """
    Gambar overlay postur pada frame gambar.
    Dipakai persis sama di Streamlit app.
    """
    COLOR_MAP = {
        'bad' : (54,  90, 216),   # BGR
        'good': (117, 158,  29),
    }
    color = COLOR_MAP.get(label, (128, 128, 128))
    h, w  = img_bgr.shape[:2]

    # Background strip atas
    overlay = img_bgr.copy()
    cv2.rectangle(overlay, (0, 0), (w, 54), (20, 20, 28), -1)
    cv2.addWeighted(overlay, 0.75, img_bgr, 0.25, 0, img_bgr)
    cv2.rectangle(img_bgr, (0, 0), (w, 54), color, 2)

    icon = '✗' if label == 'bad' else '✓'
    label_txt = f'{icon} {label.upper()}  {confidence*100:.1f}%'
    cv2.putText(img_bgr, label_txt, (12, 36),
                cv2.FONT_HERSHEY_DUPLEX, 0.9, color, 2, cv2.LINE_AA)

    # Confidence bar
    bar_w = int(w * confidence)
    cv2.rectangle(img_bgr, (0, h - 5), (w, h), (30, 30, 40), -1)
    cv2.rectangle(img_bgr, (0, h - 5), (bar_w, h), color, -1)

    # Prob kecil kanan bawah
    y_offset = h - 22
    for cls, prob in probs.items():
        c = COLOR_MAP.get(cls, (128,128,128))
        cv2.putText(img_bgr, f'{cls}: {prob*100:.0f}%',
                    (w - 110, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, c, 1, cv2.LINE_AA)
        y_offset -= 16

    return img_bgr


# Ambil beberapa gambar test untuk demo
demo_imgs = []
for cls in CLASS_NAMES:
    folder = os.path.join(DATASET_ROOT, 'test', cls)
    if os.path.isdir(folder):
        files  = [f for f in os.listdir(folder)
                  if Path(f).suffix.lower() in {'.jpg','.jpeg','.png'}]
        chosen = random.sample(files, min(4, len(files)))
        demo_imgs.extend([(os.path.join(folder, f), cls) for f in chosen])

random.shuffle(demo_imgs)
n_demo = min(8, len(demo_imgs))
n_cols_demo = 4
n_rows_demo = (n_demo + n_cols_demo - 1) // n_cols_demo

fig, axes = plt.subplots(n_rows_demo, n_cols_demo,
                         figsize=(n_cols_demo * 3.5, n_rows_demo * 3.5))
fig.suptitle('Simulasi Inference Real-time — YOLOv8 Posture Detection',
             fontsize=14, fontweight='bold')
if n_rows_demo == 1:
    axes = [axes]

for flat_idx in range(n_rows_demo * n_cols_demo):
    r, c = divmod(flat_idx, n_cols_demo)
    ax = axes[r][c]

    if flat_idx < n_demo:
        img_path, true_cls = demo_imgs[flat_idx]
        img_bgr = cv2.imread(img_path)
        img_bgr = cv2.resize(img_bgr, (280, 280))

        res        = best_model.predict(img_path, imgsz=IMG_SIZE,
                                        device=device, verbose=False)
        probs_arr  = res[0].probs.data.cpu().numpy()
        pred_idx   = int(probs_arr.argmax())
        pred_label = CLASS_NAMES[pred_idx]
        confidence = float(probs_arr[pred_idx])
        prob_dict  = dict(zip(CLASS_NAMES, probs_arr.tolist()))

        annotated = draw_posture_overlay(img_bgr.copy(), pred_label,
                                        confidence, prob_dict)
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

        ax.imshow(annotated_rgb)
        correct = pred_label == true_cls
        border  = '#1D9E75' if correct else '#D85A30'
        ax.set_title(f'True: {true_cls}  {"✓" if correct else "✗"}',
                     fontsize=9,
                     color='#1D9E75' if correct else '#D85A30')
        for spine in ax.spines.values():
            spine.set_edgecolor(border); spine.set_linewidth(2.5)
    else:
        ax.set_visible(False)

    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(WORK_DIR / 'inference_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print('Demo inference selesai!')

## Sel 14 — Perbandingan Model Final & Ringkasan

In [ ]:
# Hitung metrik nano juga untuk perbandingan
print('Evaluasi YOLOv8n-cls pada test set untuk perbandingan...')
nano_model = YOLO(NANO_PATH)
y_true_n, y_pred_n, y_probs_n, _ = run_inference_on_split(nano_model, 'test')
acc_nano = (y_true_n == y_pred_n).mean()
fpr_n, tpr_n, _ = roc_curve(y_true_n, y_probs_n[:, 0], pos_label=0)
auc_nano = auc(fpr_n, tpr_n)

# Ukuran model
size_nano  = Path(NANO_PATH).stat().st_size  / 1024 / 1024
size_small = Path(SMALL_PATH).stat().st_size / 1024 / 1024

# Tabel perbandingan
comparison_data = {
    'Model'        : ['YOLOv8n-cls', 'YOLOv8s-cls'],
    'Params'       : ['2.7M', '6.4M'],
    'Model Size'   : [f'{size_nano:.1f} MB', f'{size_small:.1f} MB'],
    'Test Accuracy': [f'{acc_nano:.4f}', f'{acc:.4f}'],
    'ROC AUC'      : [f'{auc_nano:.4f}', f'{roc_auc:.4f}'],
    'Terpilih'     : [
        '🏆' if BEST_MODEL_NAME == 'YOLOv8n-cls' else '',
        '🏆' if BEST_MODEL_NAME == 'YOLOv8s-cls' else ''
    ]
}
df_compare = pd.DataFrame(comparison_data)
print('\n=== Perbandingan Model ===')
display(df_compare)

print()
print('=' * 58)
print('  RINGKASAN PROJECT — YOLOv8 Bad Posture Detection')
print('=' * 58)
print(f'  Dataset      : {total_all} gambar (bad + good)')
print(f'  Model terbaik: {BEST_MODEL_NAME}')
print(f'  Test Accuracy: {acc*100:.2f}%')
print(f'  ROC AUC      : {roc_auc:.4f}')
print(f'  ONNX export  : {EXPORT_DIR / "posture_best.onnx"}')
print(f'  PT weight    : {EXPORT_DIR / "posture_best.pt"}')
print()
print('  Output files:')
for f in sorted(WORK_DIR.glob('*.png')):
    print(f'  ✅ {f.name}')
for f in sorted(EXPORT_DIR.iterdir()):
    print(f'  ✅ export/{f.name}')
print()
print('  Next steps:')
print('  1. Download posture_best.pt & metadata.json')
print('  2. Letakkan di folder Streamlit app')
print('  3. Ganti YOLO(\'yolov8n-cls.pt\') dengan YOLO(\'posture_best.pt\')')
print('  4. Jalankan: streamlit run app.py')
print('=' * 58)